# Colab: load a Drive checkpoint
Use your own authorized, complete model/tokenizer export. Default route: English to Twi, not Twi/French. Commit/push the updated working tree first or upload the complete updated project to `/content/TW-FR-MT`; uploading this notebook alone does not transfer package changes. Colab is interactive compute, not 24/7 hosting. [Colab FAQ](https://research.google.com/colaboratory/faq.html) warns that free runtimes without positive compute units may terminate sessions that primarily use a web UI. Sharing is off by default; do not enable it for sensitive data.

In [ ]:
from pathlib import Path
import subprocess
import os
from urllib.parse import urlparse

REPO_URL = 'https://github.com/gyasifred/TW-FR-MT.git'  # Set your pushed repository.
BRANCH = ''
REPO_DIR = Path('/content/TW-FR-MT')
parsed = urlparse(REPO_URL)
if parsed.scheme != 'https' or parsed.hostname != 'github.com' or parsed.username or parsed.password or parsed.query or parsed.fragment:
    raise ValueError('Use a credential-free HTTPS GitHub repository URL.')
if not REPO_DIR.exists():
    command = ['git', 'clone']
    if BRANCH:
        command += ['--branch', BRANCH, '--single-branch']
    subprocess.run(command + ['--', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Using existing checkout/upload unchanged:', REPO_DIR)
if (REPO_DIR / '.git').exists():
    if BRANCH:
        actual = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
        if actual != BRANCH:
            raise RuntimeError(f'Existing branch {actual!r} differs from {BRANCH!r}; resolve manually.')
    subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True)
if not (REPO_DIR / 'pyproject.toml').is_file():
    raise RuntimeError('Updated package missing: push first or upload the complete updated working tree.')
os.chdir(REPO_DIR)


In [ ]:
%pip install -e '.[train,app]'
from importlib.metadata import version
assert version('transformers') == '4.57.1', 'Use the project pin and restart the runtime if needed.'


## Mount authorized Drive storage
Open [the supplied folder](https://drive.google.com/drive/folders/13irIvPsqnryP_NJ5y6PneFKrQDJs5Qm6?usp=drive_link) using your authorized account. Add a shortcut to My Drive if necessary. Set `DRIVE_ROOT` to the actual mounted folder/shortcut path, not a name guessed from its ID. Mounting grants this notebook access to your Drive; no public download helper is used.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json
DATA_FOLDER_ID = '139FuEvqxcZzVglh0qNBIIDaXusjpgFuh'  # Training inputs; not needed to serve.
MODEL_FOLDER_ID = '13irIvPsqnryP_NJ5y6PneFKrQDJs5Qm6'
DRIVE_ROOT = Path('/content/drive/MyDrive/TW-FR-MT')  # Choose the actual mounted model folder.
MODEL_ROOT = DRIVE_ROOT
CHECKPOINT_RELATIVE = 'runs/en-tw-run-001/final'  # Set your actual English -> Twi export.
REGISTRY_PATH = DRIVE_ROOT / 'registries/en-tw.json'
DEVICE = 'auto'
ENABLE_PUBLIC_SHARE = False  # True creates a public Gradio link; opt in deliberately.
LAUNCH_UI = False  # Opt in only where your Colab plan/policies permit this interactive use.
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError('Set DRIVE_ROOT to the actual mounted folder/shortcut path.')
checkpoint = (MODEL_ROOT / CHECKPOINT_RELATIVE).resolve()
if Path(CHECKPOINT_RELATIVE).is_absolute() or not checkpoint.is_relative_to(MODEL_ROOT.resolve()):
    raise ValueError('Checkpoint must be relative to and stay inside MODEL_ROOT.')
if not checkpoint.is_dir() or not (checkpoint / 'config.json').is_file():
    raise FileNotFoundError(f'Choose a complete saved model/tokenizer export: {checkpoint}')
registry = {'routes': [{'id': 'en-tw', 'label': 'English to Twi', 'source_lang': 'en', 'target_lang': 'tw', 'steps': [
    {'source_lang': 'en', 'target_lang': 'tw', 'checkpoint': CHECKPOINT_RELATIVE,
     'source_tokenizer_lang': None, 'target_tokenizer_lang': None, 'source_prefix': ''}
]}]}
REGISTRY_PATH.parent.mkdir(parents=True, exist_ok=True)
payload = json.dumps(registry, indent=2) + '\n'
if REGISTRY_PATH.exists():
    if REGISTRY_PATH.read_text(encoding='utf-8') != payload:
        raise FileExistsError('Registry differs; choose a new REGISTRY_PATH instead of overwriting it.')
else:
    with REGISTRY_PATH.open('x', encoding='utf-8') as handle:
        handle.write(payload)
print(REGISTRY_PATH)
print(payload)


## Other directions require real checkpoints
For Twi to French, explicitly supply a model trained for that pair and change the route and step languages. A schema-only example is below; it does not create or download a model. For a pivot, provide both compatible checkpoints as ordered steps, with the first target matching the next source. Verify tokenizer codes/prefixes from the actual model; do not fabricate `tw`/`fr` special tokens. Never relabel the English-to-Twi export as Twi-to-French. No weights are copied or uploaded automatically.
```json
{"routes":[{"id":"tw-fr","label":"Twi to French","source_lang":"tw","target_lang":"fr","steps":[{"source_lang":"tw","target_lang":"fr","checkpoint":"tw-fr/final","source_tokenizer_lang":null,"target_tokenizer_lang":null,"source_prefix":""}]}]}
```


In [ ]:
from twfr_mt.ui import create_app
app = create_app(registry_path=str(REGISTRY_PATH), model_root=str(MODEL_ROOT), device=DEVICE)
if LAUNCH_UI:
    app.launch(share=ENABLE_PUBLIC_SHARE, max_file_size='1mb', enable_monitoring=False, show_error=False, blocked_paths=[str(MODEL_ROOT.resolve())])
else:
    print('App constructed; UI launch and public sharing remain disabled.')


Stop the app with `app.close()` when done. Public sharing exposes the app to people with the link; keep confidential text and credentials out of inputs and outputs. Runtime termination removes `/content` files, not verified Drive exports. Loading checkpoints from Drive can be slow; see `docs/COLAB.md` for I/O caveats and `docs/DEPLOYMENT.md` for authenticated export to a real hosting environment.